# Top-10 filter ranking versus the metaheuristic signatures

Equation (11) averages per-gene scores. For aMI, aSU, aGR and aCHI2 a gene's score does not depend on the other
genes, so with a minimum length of 10 genes the 10 best-ranked genes maximize Eq. (11): no search can do better on the
objective. aReliefF is the exception, because a gene's weight depends on the other genes. This notebook compares each
metaheuristic signature with that ranking, first on the objective and then on the test set.

**Design.** Each of the 15 finalists (one per metaheuristic and objective, with the classifier of Table 8) keeps its
classifier, parameters and seed; only its gene signature is replaced by the top 10 of the same objective. Both versions
are trained on the training partition and tested on the same 30 patients. Nothing else is re-run: no metaheuristic
search, no cross-validation, no classifier selection. The finalists' test results were already known, so the
comparison is exploratory, and its configurations are frozen in git before the new test results are computed.

**Code.** All the logic is in this notebook. The top 10 is selected by Weka's `Ranker` with the same evaluators as
the objective functions. From the repository the notebook uses only `ofunctions.allObjectiveFunctionWeka`, the
objective function the metaheuristics optimized, to compute Eq. (11) and to check that both use the same scores. The LaTeX
table is written afterwards by `python latex_tables/top10_ranking_vs_mh_table.py`.

**Run all cells from the top.** Settings are in section 1. Genes are 0-based column indices in the files and *g*N
(N = index + 1) in the manuscript.

| Section | Stage | What it does | Test set |
|---|---|---|---|
| 2 | | Every Weka computation: Weka's `Ranker`, gene scores and Eq. (11) merits | No |
| 3 | 1 | Ranking and top 10 of each objective on the training partition | No |
| 4 | 2 | Each signature against its top 10, and the configuration file of the 30 models | No |
| 5 | | Test data, and a check of the procedure on the 15 finalists, whose results are already known | Known results only |
| 6 | Freeze | Stops unless the configuration file is committed in git and unchanged | |
| 7 | 3 | Test evaluation of the top-10 models, 95% confidence intervals and paired differences | Yes |

## 1. Setup and settings

In [1]:
import ast
import hashlib
import json
import logging
import os
import re
import subprocess
import warnings

import numpy as np
import pandas as pd
import sklearn
from scipy.stats import binomtest, false_discovery_control
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# All paths are relative to the repository root, the folder that holds this notebook.
# VS Code gives the notebook's own path in __vsc_ipynb_file__; plain Jupyter already starts the kernel there.
if '__vsc_ipynb_file__' in globals():
    os.chdir(os.path.dirname(__vsc_ipynb_file__))
assert os.path.isdir('data') and os.path.isfile('ofunctions.py'), (
    f'Start the kernel in the repository root: the working directory is {os.getcwd()}')

# SVC(probability=True) is the classifier of the finalists; recent scikit-learn versions warn that the parameter is deprecated
warnings.filterwarnings('ignore', message='The `probability` parameter was deprecated', category=FutureWarning)

pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)

In [2]:
# ---- Data (in data/) ----
TRAIN_FILE = 'train_normal_rtf.csv'        # training partition, 356 rows, scaled: ranking and classifier training
FULL_FILE = 'normal_rtf.csv'               # the 396 rows the metaheuristics searched (training + test rows); stage 2 only
TEST_FILE = 'test_rtf_v2_cleaned.csv'      # the 30 test patients (20 negative, 10 positive), raw scale
TEST_IDS_FILE = 'test_rtf_with_ID_original.csv'      # the same 30 patients, in the same order, with their SampleID
SCALING_REFERENCE = 'rtfDataSet.csv'                 # raw data whose column ranges scaled the training data
SCALED_TEST_FILE = 'test_rtf_v2_cleaned_normal.csv'  # the test patients scaled by test_classifiers.ipynb (for a check)

# ---- Finalists ----
# One per metaheuristic and objective: signature (0-based genes) and classifier chosen in Table 8
FINALISTS_FILE = 'experiment_results/R3_v2/best_results_step_R3_v2.csv'
# Their published test results (test_classifiers.ipynb): the re-run in section 5 must reproduce these counts
KNOWN_RESULTS_FILE = 'experiment_results/R3_v2/test_results_15_schemes.csv'

# ---- Experiment ----
META_ORDER = ['DE', 'GA', 'SA']
OBJECTIVES = ['mi', 'sim_uncertainty', 'gain_ratio', 'relieff', 'chi2']          # in the manuscript order
OBJECTIVE_LABELS = {'mi': 'aMI', 'sim_uncertainty': 'aSU', 'gain_ratio': 'aGR', 'relieff': 'aReliefF', 'chi2': 'aCHI2'}
SEPARABLE_OBJECTIVES = ['mi', 'sim_uncertainty', 'gain_ratio', 'chi2']           # the top K maximizes Eq. (11)
K = 10                      # genes kept: the minimum subset length of the metaheuristics
RF_SEED = 45                # random_state of the random forests, as in test_classifiers.ipynb

# ---- Confidence intervals (same method and resamples as confidence_intervals.py) ----
CONFIDENCE = 0.95
RESAMPLES = 100_000
BOOTSTRAP_SEED = 45
METRICS = ['F1', 'Sn', 'Sp', 'ACC', 'PRE']
DIFFERENCE_METRICS = ['F1', 'Sn', 'Sp', 'ACC']

# ---- Outputs (overwritten on every run; the columns are explained after each stage) ----
OUTPUT_DIR = 'experiment_results/top10_ranking_vs_mh'
RANKINGS_FILE = f'{OUTPUT_DIR}/rankings.csv'                  # stage 1
TOP10_FILE = f'{OUTPUT_DIR}/top10_subsets.csv'                # stage 1
COMPARISON_FILE = f'{OUTPUT_DIR}/signatures_vs_top10.csv'     # stage 2
PAIRS_FILE = f'{OUTPUT_DIR}/top10_pairs.csv'                  # stage 2: the frozen configurations
PREDICTIONS_FILE = f'{OUTPUT_DIR}/test_predictions.csv'       # stage 3
METRICS_FILE = f'{OUTPUT_DIR}/test_metrics.csv'               # stage 3 (input of the LaTeX table)
DIFFERENCES_FILE = f'{OUTPUT_DIR}/test_differences.csv'       # stage 3
RUN_FILE = f'{OUTPUT_DIR}/test_run.json'                      # stage 3: freeze commit, versions and settings

# ---- Java runtime for Weka ----
# There is no system Java on this machine; any Java 11 or later works. A JAVA_HOME already set takes precedence.
JAVA_HOME = '/opt/JetBrains Rider-2026.1.2/jbr'

## 2. Weka computations

The Weka JVM can be started only once per kernel, so every computation that needs Weka is in this section, and the
stages below only use its results. To run this section again, restart the kernel.

* **Top 10 with Weka's `Ranker`.** For aSU, aGR, aReliefF and aCHI2, Weka's attribute selection runs with the evaluator
  of the objective function (`SymmetricalUncertAttributeEval`, `GainRatioAttributeEval`, `ReliefFAttributeEval`,
  `ChiSquaredAttributeEval`) and the `Ranker` search with `-N 10`: Weka scores the 70 genes, ranks them and selects
  the 10 best. The data are prepared as in `ofunctions.py`, which the metaheuristics used: the label is made nominal
  with `NumericToNominal`. `Ranker` also returns the whole ranking, with every gene's score.
* **aMI.** In this project mutual information is computed with scikit-learn (`ofunctions.py`), not with Weka, so there
  is no Weka evaluator for it. Its 70 scores are sorted, highest first, and the first 10 are kept, which is what
  `Ranker` does.
* **Check.** For every objective, the scores of the ranking must equal those of
  `ofunctions.allObjectiveFunctionWeka(..., get_mean=False)`, the function the metaheuristics optimized: both use the
  same evaluators and options.
* **Merits.** Eq. (11) of a subset is `allObjectiveFunctionWeka(subset, ...)` with its default `get_mean=True`. It is
  computed for every top 10 and every finalist's signature.

Everything is computed on the training partition, which gives the top 10 of the experiment, and on the 396-row file
the metaheuristics searched, which gives the context of stage 2.

In [3]:
WEKA_EVALUATORS = {
    'sim_uncertainty': 'weka.attributeSelection.SymmetricalUncertAttributeEval',
    'gain_ratio': 'weka.attributeSelection.GainRatioAttributeEval',
    'relieff': 'weka.attributeSelection.ReliefFAttributeEval',
    'chi2': 'weka.attributeSelection.ChiSquaredAttributeEval',
}


def weka_ranker(X, y, of, n):
    # Weka's attribute selection with the evaluator of `of` and the Ranker search, selecting the n best genes.
    # The data are prepared as in ofunctions.py: the label, the last attribute, is made nominal
    dataset = create_instances_from_matrices(X, y, name='generated from matrices')
    nominal = Filter(classname='weka.filters.unsupervised.attribute.NumericToNominal', options=['-R', 'last'])
    nominal.inputformat(dataset)
    data = nominal.filter(dataset)
    selection = AttributeSelection()
    selection.search(ASSearch(classname='weka.attributeSelection.Ranker', options=['-N', str(n)]))
    selection.evaluator(ASEvaluation(classname=WEKA_EVALUATORS[of]))
    selection.select_attributes(data)
    # selected_attributes ends with the class attribute, which is not a gene
    selected = sorted(int(i) for i in selection.selected_attributes if int(i) != data.class_index)
    # ranked_attributes: every gene as (index, score), best first
    ranking = [(int(gene), float(score)) for gene, score in selection.ranked_attributes]
    return selected, ranking


def mi_ranking(X, y):
    # aMI: the scikit-learn scores of ofunctions.py, sorted by score, highest first (equal scores: lower gene index)
    scores = ofunctions.allObjectiveFunctionWeka(list(range(X.shape[1])), X, y, type_of='mi', get_mean=False)
    order = sorted(range(len(scores)), key=lambda gene: (-scores[gene], gene))
    return [(gene, float(scores[gene])) for gene in order]


def reference_scores(X, y, of):
    # The scores of ofunctions.allObjectiveFunctionWeka, the function the metaheuristics optimized, in column order
    result = np.asarray(ofunctions.allObjectiveFunctionWeka(list(range(X.shape[1])), X, y, type_of=of, get_mean=False),
                        dtype=float)
    if of == 'mi':
        return result
    scores = np.full(X.shape[1], np.nan)
    scores[result[:, 0].astype(int)] = result[:, 1]
    return scores


def tie_rule_top(ranking, k):
    # The fixed tie rule, used only if the k-th and (k+1)-th genes have the same score: the lower gene index enters
    cut = ranking[k - 1][1]
    above = [gene for gene, score in ranking if score > cut]
    tied = sorted(gene for gene, score in ranking if score == cut)
    return sorted(above + tied[:k - len(above)])


def load(name):
    data = pd.read_csv(f'data/{name}')
    return data.drop(columns=['label']).to_numpy(), data['label'].to_numpy(dtype=np.intp)

In [4]:
os.environ.setdefault('JAVA_HOME', JAVA_HOME)
import weka.core.jvm as jvm
from weka.attribute_selection import ASEvaluation, ASSearch, AttributeSelection
from weka.core.dataset import create_instances_from_matrices
from weka.filters import Filter

import ofunctions

DATA = {TRAIN_FILE: load(TRAIN_FILE), FULL_FILE: load(FULL_FILE)}
for name, (X, y) in DATA.items():
    print(f'{name}: {X.shape[0]} rows, {X.shape[1]} genes')

# The finalists in the manuscript order, numbered as pairs 1-15. Genes keep the order of the file: random forests
# depend on the column order, and the published results used this order (see section 5)
finalists = pd.read_csv(FINALISTS_FILE)
finalists['Subset'] = finalists['Subset'].map(ast.literal_eval)
finalists['order'] = finalists['Meta'].map(META_ORDER.index) * len(OBJECTIVES) + finalists['OF'].map(OBJECTIVES.index)
finalists = finalists.sort_values('order').drop(columns='order').reset_index(drop=True)
finalists.insert(0, 'Pair', range(1, len(finalists) + 1))
finalists.insert(1, 'Finalist', finalists['Meta'] + ' + ' + finalists['OF'].map(OBJECTIVE_LABELS))
assert len(finalists) == len(META_ORDER) * len(OBJECTIVES), 'expected one finalist per metaheuristic and objective'

rankings_by = {}    # rankings_by[file, of]: every gene as (index, score), best first
scores = {}         # scores[file, of]: one score per gene, in column order
top_genes = {}      # top_genes[file, of]: the top 10, ascending
tie_at_cut = {}     # tie_at_cut[file, of]: the 10th and 11th genes have the same score
merits = {}         # merits[file, of, genes]: Eq. (11) of the subset `genes` on `file`


def merit(file, of, genes):
    key = (file, of, tuple(genes))
    if key not in merits:
        X, y = DATA[file]
        merits[key] = ofunctions.allObjectiveFunctionWeka(list(genes), X, y, type_of=of)
    return merits[key]


jvm.start(packages=True, logging_level=logging.WARNING)
try:
    for file, (X, y) in DATA.items():
        for of in OBJECTIVES:
            if of == 'mi':
                ranking = mi_ranking(X, y)
                selected = sorted(gene for gene, _ in ranking[:K])
            else:
                selected, ranking = weka_ranker(X, y, of, K)
            score = np.full(X.shape[1], np.nan)
            for gene, value in ranking:
                score[gene] = value
            # Check: the same scores as the objective function the metaheuristics optimized
            assert np.allclose(score, reference_scores(X, y, of), rtol=0, atol=1e-12), f'{of}: scores differ from ofunctions'
            tie_at_cut[file, of] = ranking[K - 1][1] == ranking[K][1]
            rankings_by[file, of], scores[file, of] = ranking, score
            top_genes[file, of] = tie_rule_top(ranking, K) if tie_at_cut[file, of] else selected
    for file in DATA:
        for of in OBJECTIVES:
            merit(file, of, top_genes[TRAIN_FILE, of])     # the top 10 of the experiment, on both files
            merit(file, of, top_genes[file, of])           # the top 10 of each file, on that file
        for _, finalist in finalists.iterrows():
            merit(file, finalist['OF'], finalist['Subset'])   # the signatures, on both files
finally:
    # Files are written after the JVM stops: while it runs, JPype's import hook can break the lazy imports
    # pandas does when writing files
    jvm.stop()
print(f'{len(rankings_by)} rankings and {len(merits)} merits computed. Check passed: the ranking scores equal '
      'those of ofunctions.allObjectiveFunctionWeka for every objective and file.')

train_normal_rtf.csv: 356 rows, 70 genes
normal_rtf.csv: 396 rows, 70 genes



Nov 27, 2026 7:56:31 PM com.github.fommil.netlib.ARPACK <clinit>
Nov 27, 2026 7:56:31 PM com.github.fommil.netlib.ARPACK <clinit>


10 rankings and 39 merits computed. Check passed: the ranking scores equal those of ofunctions.allObjectiveFunctionWeka for every objective and file.


## 3. Stage 1: ranking on the training partition

For aSU, aGR, aReliefF and aCHI2, the top 10 is the selection of Weka's `Ranker` (`-N 10`) on the training
partition; for aMI it is the 10 highest scores. Its merit is Eq. (11) on those 10 genes. For the separable objectives
the merit must equal the mean of the 10 scores, and the cell checks it. For aReliefF it does not, because the weights
change with the other genes passed to the evaluator: its top 10 is a ranking, not the maximum of Eq. (11).

**Tie rule.** Weka's `Ranker` has no fixed order for genes with exactly the same score: in a test with identical genes,
their order depended on their positions. So a fixed rule is added: if the 10th and 11th genes have the same score, the
one with the lower gene index (lower *g* number) enters the top 10. The rule is arbitrary (a lower gene number means
nothing biologically), but it is fixed in advance. It only matters for a tie at the cut; a tie inside the top 10 only
changes the order of genes that are selected anyway. `Tie_at_cut` in `top10_subsets.csv` shows whether it happened.

In [5]:
EVALUATOR_NAMES = {of: name.rsplit('.', 1)[1] for of, name in WEKA_EVALUATORS.items()}
rankings, top10 = [], []
for of in OBJECTIVES:
    ranking, score = rankings_by[TRAIN_FILE, of], scores[TRAIN_FILE, of]
    for rank, (gene, value) in enumerate(ranking, start=1):
        rankings.append({'OF': of, 'Rank': rank, 'Gene': gene, 'Gene_label': f'g{gene + 1}',
                         'Score': value, 'In_top10': gene in top_genes[TRAIN_FILE, of]})
    top = top_genes[TRAIN_FILE, of]
    method = (f'Weka Ranker -N {K} with {EVALUATOR_NAMES[of]}' if of in WEKA_EVALUATORS
              else 'scikit-learn mutual information, sorted')
    top10.append({
        'Meta': f'TOP{K}', 'OF': of, 'IT': 0, 'Subset': str(top), 'Length': len(top),
        'Settings': f'{method} on {TRAIN_FILE}; ties at the cut to the lower gene index',
        'Merit': merit(TRAIN_FILE, of, top),
        'Mean_of_scores': score[top].mean(),
        f'Score_{K}th': ranking[K - 1][1], f'Score_{K + 1}th': ranking[K][1],
        'Tie_at_cut': bool(tie_at_cut[TRAIN_FILE, of]),
    })
rankings, top10 = pd.DataFrame(rankings), pd.DataFrame(top10)

# Check: for the separable objectives, Eq. (11) on the top 10 is the mean of their scores
separable = top10[top10['OF'].isin(SEPARABLE_OBJECTIVES)]
assert np.allclose(separable['Merit'], separable['Mean_of_scores'], rtol=0, atol=1e-9), 'merit differs from the mean of the scores'
print('Check passed: for', ', '.join(OBJECTIVE_LABELS[of] for of in SEPARABLE_OBJECTIVES),
      'the merit equals the mean of the 10 scores.')

os.makedirs(OUTPUT_DIR, exist_ok=True)
rankings.to_csv(RANKINGS_FILE, index=False)
top10.to_csv(TOP10_FILE, index=False)
print('Written to', RANKINGS_FILE, 'and', TOP10_FILE)

view = top10.assign(Function=top10['OF'].map(OBJECTIVE_LABELS),
                    Genes=[', '.join(f'g{g + 1}' for g in ast.literal_eval(s)) for s in top10['Subset']])
display(view[['Function', 'Length', 'Genes', 'Merit', 'Mean_of_scores', f'Score_{K}th', f'Score_{K + 1}th', 'Tie_at_cut']])
ties = view[view['Tie_at_cut']]
print('The tie rule decided the last gene for:', ', '.join(ties['Function']) if len(ties) else 'no objective')

Check passed: for aMI, aSU, aGR, aCHI2 the merit equals the mean of the 10 scores.
Written to experiment_results/top10_ranking_vs_mh/rankings.csv and experiment_results/top10_ranking_vs_mh/top10_subsets.csv


,Function,Length,Genes,Merit,Mean_of_scores,Score_10th,Score_11th,Tie_at_cut
0,aMI,10,"g10, g11, g22, g24, g36, g51, g53, g60, g64, g66",0.091163,0.091163,0.071480,0.071240,False
1,aSU,10,"g10, g15, g22, g33, g35, g43, g51, g64, g66, g68",0.085949,0.085949,0.071896,0.069957,False
2,aGR,10,"g15, g22, g34, g35, g43, g45, g51, g59, g64, g66",0.109775,0.109775,0.085603,0.078466,False
3,aReliefF,10,"g3, g10, g24, g25, g29, g33, g38, g51, g58, g60",0.012966,0.016771,0.013657,0.013562,False
4,aCHI2,10,"g10, g12, g13, g15, g33, g51, g53, g64, g66, g68",37.883813,37.883813,29.757752,28.353752,False


The tie rule decided the last gene for: no objective


### Stage 1 output columns

`rankings.csv`: one row per objective and gene (5 × 70 = 350 rows), sorted by objective and rank.

| Column | Meaning |
|---|---|
| `OF` | Objective: `mi` (aMI), `sim_uncertainty` (aSU), `gain_ratio` (aGR), `relieff` (aReliefF), `chi2` (aCHI2) |
| `Rank` | Position in the ranking, 1 = highest score: the order of Weka's `Ranker` (aSU, aGR, aReliefF, aCHI2) or of the sorted scores (aMI) |
| `Gene`, `Gene_label` | 0-based column index in the data files; name *g*N in the manuscript (N = `Gene` + 1) |
| `Score` | Score of the gene on the training partition (Ω*i* in Eq. 11), computed with all 70 genes passed to the evaluator |
| `In_top10` | `True` for the 10 best-ranked genes |

`top10_subsets.csv`: one row per objective. The first seven columns follow the layout of the metaheuristic subset files.

| Column | Meaning |
|---|---|
| `Meta`, `IT` | `TOP10`: the subset comes from the ranking; `IT` = 0 because there is no search |
| `Subset`, `Length` | The 10 genes (0-based, ascending) and their number |
| `Settings` | How the top 10 was selected (Weka `Ranker` and its evaluator, or the aMI sort), data file and tie rule |
| `Merit` | Eq. (11) on the training partition when the evaluator receives only these 10 genes |
| `Mean_of_scores` | Mean of the 10 genes' `Score`; equal to `Merit` except for aReliefF |
| `Score_10th`, `Score_11th` | Score of the last gene in the top 10 and of the best gene left out |
| `Tie_at_cut` | `True` if those two scores are equal, so the tie rule decided which gene entered |

**Note.** `select_final_model.py` also reads `top10_subsets.csv` (columns Meta, OF and Subset), as comparators that
get their own classifier from the cross-validation results. This experiment instead keeps each finalist's classifier.

## 4. Stage 2: each signature against its top 10

For each finalist, its signature is compared with the top 10 of its objective:

* **Genes.** Number in common, Jaccard index, and the genes found in only one of the two.
* **Objective.** Eq. (11) of both on the training partition. For the separable objectives the top 10 is the maximum
  there, so its merit must be at least the signature's, and the cell checks it. On the 396-row file the metaheuristics
  searched, the cell also gives the maximum of Eq. (11) (the merit of that file's own top 10) and whether the signature
  reached it.

Then it writes `top10_pairs.csv`, the configuration of the 30 models to test: for each finalist, its signature and its
top 10, with the same classifier, parameters and seed. A signature keeps the gene order of the finalists file and a
top 10 is in ascending order (the order matters for random forests, see section 5). A top-10 model shared by several
finalists (same objective, classifier and parameters) has one model name and is trained and tested once.

In [6]:
comparison = []
for _, finalist in finalists.iterrows():
    of = finalist['OF']
    mh, top = finalist['Subset'], top_genes[TRAIN_FILE, of]
    common = set(mh) & set(top)
    separable = of in SEPARABLE_OBJECTIVES
    max_full = merit(FULL_FILE, of, top_genes[FULL_FILE, of]) if separable else np.nan
    comparison.append({
        'Pair': finalist['Pair'], 'Finalist': finalist['Finalist'], 'Meta': finalist['Meta'], 'OF': of,
        'Classifier': finalist['Classifier'], 'Length_MH': len(mh), 'Length_top10': len(top),
        'Common': len(common), 'Jaccard': len(common) / len(set(mh) | set(top)),
        'Only_MH': ', '.join(f'g{g + 1}' for g in sorted(mh) if g not in common),
        'Only_top10': ', '.join(f'g{g + 1}' for g in top if g not in common),
        'Merit_MH_train': merit(TRAIN_FILE, of, mh), 'Merit_top10_train': merit(TRAIN_FILE, of, top),
        'Top10_at_least_MH': (merit(TRAIN_FILE, of, top) >= merit(TRAIN_FILE, of, mh) - 1e-12) if separable else None,
        'Merit_MH_full': merit(FULL_FILE, of, mh), 'Max_full': max_full,
        'MH_at_max_full': bool(abs(merit(FULL_FILE, of, mh) - max_full) <= 1e-9) if separable else None,
    })
comparison = pd.DataFrame(comparison)

checked = comparison[comparison['OF'].isin(SEPARABLE_OBJECTIVES)]
assert checked['Top10_at_least_MH'].all(), 'a signature beats the top 10 of a separable objective: check the scoring'
print(f'Check passed: on the training partition the top 10 has at least the merit of all {len(checked)} signatures '
      'of the separable objectives.')
at_max = checked[checked['MH_at_max_full'].astype(bool)]
print(f'On the 396-row file the metaheuristics searched, {len(at_max)} of {len(checked)} signatures reached the maximum '
      f'of Eq. (11): {", ".join(at_max["Finalist"]) or "none"}.')
display(comparison.drop(columns=['Meta', 'OF']))

Check passed: on the training partition the top 10 has at least the merit of all 12 signatures of the separable objectives.
On the 396-row file the metaheuristics searched, 4 of 12 signatures reached the maximum of Eq. (11): DE + aMI, DE + aSU, DE + aGR, DE + aCHI2.


,Pair,Finalist,Classifier,Length_MH,Length_top10,Common,Jaccard,Only_MH,Only_top10,Merit_MH_train,Merit_top10_train,Top10_at_least_MH,Merit_MH_full,Max_full,MH_at_max_full
0,1,DE + aMI,Naive,10,10,7,0.538462,"g4, g58, g68","g22, g24, g36",0.084706,0.091163,True,0.082621,0.082621,True
1,2,DE + aSU,RF estimators: 60,10,10,7,0.538462,"g12, g45, g59","g33, g35, g68",0.083722,0.085949,True,0.078548,0.078548,True
2,3,DE + aGR,RF estimators: 60,10,10,10,1.000000,,,0.109775,0.109775,True,0.103818,0.103818,True
3,4,DE + aReliefF,Knn k = 3,18,10,9,0.473684,"g7, g17, g19, g21, g23, g39, g49, g50, g63",g60,0.013992,0.012966,None,0.014495,NaN,None
4,5,DE + aCHI2,RF estimators: 100,10,10,7,0.538462,"g39, g50, g59","g10, g13, g33",35.447290,37.883813,True,37.181303,37.181303,True
5,6,GA + aMI,RF estimators: 60,10,10,5,0.333333,"g4, g8, g33, g68, g70","g11, g22, g24, g36, g66",0.077305,0.091163,True,0.078270,0.082621,False
6,7,GA + aSU,RF estimators: 80,10,10,6,0.428571,"g13, g20, g34, g52","g10, g15, g22, g33",0.078755,0.085949,True,0.071600,0.078548,False
7,8,GA + aGR,RF estimators: 80,10,10,6,0.428571,"g16, g17, g44, g68","g15, g22, g51, g59",0.097563,0.109775,True,0.090621,0.103818,False
8,9,GA + aReliefF,Knn k = 3,11,10,5,0.312500,"g7, g21, g23, g39, g50, g68","g10, g24, g25, g33, g60",0.012852,0.012966,None,0.013328,NaN,None
9,10,GA + aCHI2,SVM (Poly),10,10,6,0.428571,"g20, g39, g45, g59","g13, g15, g33, g66",33.142945,37.883813,True,34.505093,37.181303,False


In [7]:
def make_classifier(label):
    # The classifier of a finalist, built as in test_classifiers.ipynb
    text = label.strip().lower()
    number = re.search(r'\d+', label)
    if text.startswith('rf'):
        return RandomForestClassifier(int(number.group()), random_state=RF_SEED)
    if text.startswith('knn'):
        return KNeighborsClassifier(int(number.group()))
    if text.startswith('naive') or text == 'nb':
        return GaussianNB()
    if text.startswith('svm'):
        return SVC(kernel='poly', probability=True)
    if text == 'lda':
        return LinearDiscriminantAnalysis()
    raise ValueError(f'Unknown classifier: {label!r}')


def seed_note(label):
    text = label.strip().lower()
    if text.startswith('rf'):
        return f'random_state={RF_SEED}'
    if text.startswith('svm'):
        return 'none: the probability estimates are not seeded; the predicted labels do not depend on them'
    return 'none: deterministic'


pairs = []
for _, finalist in finalists.iterrows():
    of, label = finalist['OF'], finalist['Classifier']
    mh, top = finalist['Subset'], top_genes[TRAIN_FILE, of]
    for signature, genes in (('MH', mh), ('TOP10', top)):
        classifier = make_classifier(label)
        pairs.append({
            'Pair': finalist['Pair'], 'Finalist': finalist['Finalist'], 'Meta': finalist['Meta'], 'OF': of,
            'Signature': signature,
            'Model': f"{finalist['Meta']}-{of}-{label}" if signature == 'MH' else f'TOP{K}-{of}-{label}',
            'Subset': str(genes), 'Length': len(genes), 'Classifier': label,
            'Estimator': type(classifier).__name__,
            'Parameters': json.dumps(classifier.get_params(), sort_keys=True, default=str),
            'Seed': seed_note(label),
            'Same_as_MH': signature == 'TOP10' and sorted(genes) == sorted(mh),
        })
pairs = pd.DataFrame(pairs)

# Top-10 models used by more than one finalist: same genes, classifier and parameters, trained and tested once
users = pairs[pairs['Signature'] == 'TOP10'].groupby('Model')['Finalist'].apply(list)
pairs['Shared_by'] = [('; '.join(users[model]) if signature == 'TOP10' and len(users[model]) > 1 else '')
                      for model, signature in zip(pairs['Model'], pairs['Signature'])]
assert (pairs.groupby('Model')[['Subset', 'Parameters']].nunique() == 1).all().all(), 'a model name with two configurations'

comparison.to_csv(COMPARISON_FILE, index=False)
pairs.to_csv(PAIRS_FILE, index=False)
print('Written to', COMPARISON_FILE, 'and', PAIRS_FILE)
print(f"{len(pairs)} rows, {pairs['Model'].nunique()} distinct models "
      f"({(pairs['Signature'] == 'MH').sum()} signatures, {pairs.loc[pairs['Signature'] == 'TOP10', 'Model'].nunique()} top-10 models)")
display(pairs[['Pair', 'Finalist', 'Signature', 'Model', 'Length', 'Estimator', 'Seed', 'Same_as_MH', 'Shared_by']])

Written to experiment_results/top10_ranking_vs_mh/signatures_vs_top10.csv and experiment_results/top10_ranking_vs_mh/top10_pairs.csv
30 rows, 26 distinct models (15 signatures, 11 top-10 models)


,Pair,Finalist,Signature,Model,Length,Estimator,Seed,Same_as_MH,Shared_by
0,1,DE + aMI,MH,DE-mi-Naive,10,GaussianNB,none: deterministic,False,
1,1,DE + aMI,TOP10,TOP10-mi-Naive,10,GaussianNB,none: deterministic,False,
2,2,DE + aSU,MH,DE-sim_uncertainty-RF estimators: 60,10,RandomForestClassifier,random_state=45,False,
3,2,DE + aSU,TOP10,TOP10-sim_uncertainty-RF estimators: 60,10,RandomForestClassifier,random_state=45,False,DE + aSU; SA + aSU
4,3,DE + aGR,MH,DE-gain_ratio-RF estimators: 60,10,RandomForestClassifier,random_state=45,False,
5,3,DE + aGR,TOP10,TOP10-gain_ratio-RF estimators: 60,10,RandomForestClassifier,random_state=45,True,
6,4,DE + aReliefF,MH,DE-relieff-Knn k = 3,18,KNeighborsClassifier,none: deterministic,False,
7,4,DE + aReliefF,TOP10,TOP10-relieff-Knn k = 3,10,KNeighborsClassifier,none: deterministic,False,DE + aReliefF; GA + aReliefF
8,5,DE + aCHI2,MH,DE-chi2-RF estimators: 100,10,RandomForestClassifier,random_state=45,False,
9,5,DE + aCHI2,TOP10,TOP10-chi2-RF estimators: 100,10,RandomForestClassifier,random_state=45,False,DE + aCHI2; SA + aCHI2


### Stage 2 output columns

`signatures_vs_top10.csv`: one row per finalist (15 rows).

| Column | Meaning |
|---|---|
| `Pair`, `Finalist` | Number of the pair (manuscript order: DE, GA, SA; aMI, aSU, aGR, aReliefF, aCHI2) and its name |
| `Meta`, `OF`, `Classifier` | Metaheuristic, objective and classifier of the finalist (Table 8) |
| `Length_MH`, `Length_top10` | Number of genes of the signature and of the top 10 |
| `Common`, `Jaccard` | Genes in both; Jaccard index (genes in both / genes in either) |
| `Only_MH`, `Only_top10` | Genes in only one of the two, as *g*N |
| `Merit_MH_train`, `Merit_top10_train` | Eq. (11) of the signature and of the top 10 on the training partition |
| `Top10_at_least_MH` | Separable objectives: the top 10 has at least the signature's merit (it is the maximum). Empty for aReliefF |
| `Merit_MH_full` | Eq. (11) of the signature on the 396-row file the metaheuristics searched |
| `Max_full` | Separable objectives: maximum of Eq. (11) on that file, the merit of its own top 10 |
| `MH_at_max_full` | Separable objectives: the signature reached that maximum |

`top10_pairs.csv`: the frozen configurations, two rows per finalist (30 rows).

| Column | Meaning |
|---|---|
| `Pair`, `Finalist`, `Meta`, `OF` | As above |
| `Signature` | `MH`: the finalist's signature; `TOP10`: the top 10 of its objective |
| `Model` | Name of the model. A top-10 model shared by several finalists has one name and is evaluated once |
| `Subset`, `Length` | Genes (0-based) in the order the model receives them, and their number. `MH`: the order of the finalists file; `TOP10`: ascending |
| `Classifier` | The finalist's classifier, as written in the finalists file |
| `Estimator`, `Parameters` | scikit-learn class and all its parameters (`get_params()`), as JSON |
| `Seed` | Seed of the classifier, or why it has none |
| `Same_as_MH` | `TOP10` rows: the top 10 is the signature itself, so both rows are the same model |
| `Shared_by` | `TOP10` rows: the finalists that share this model, when there is more than one |

## 5. Test data and a check of the procedure

The 30 test patients are read on the raw scale and scaled with the column ranges of `rtfDataSet.csv`, the ranges that
scaled the training data, as in `test_classifiers.ipynb`; the cell checks the result against
`test_rtf_v2_cleaned_normal.csv`. Patients are identified by their `SampleID` from `test_rtf_with_ID_original.csv`,
which holds the same rows in the same order (checked).

Each model is trained on the training partition restricted to its genes and then predicts the test patients. Before
the freeze, only the 15 metaheuristic models are run. Their test results are already published, and the cell checks
that the confusion counts are exactly those of `test_results_15_schemes.csv`: the procedure reproduces
`test_classifiers.ipynb` without computing any new test result.

**Gene order.** A random forest's result depends on the order of its genes (columns), even with a fixed seed, because
the seed fixes the sequence of random choices, not which gene each choice falls on. The published results used the
gene order of the finalists file, so the signatures keep it. The last cell of this section shows how much the four
random forests whose genes are not in ascending order change when only that order changes: a measure of how large a
difference an arbitrary choice can produce.

In [8]:
raw_test = pd.read_csv(f'data/{TEST_FILE}')
gene_columns = [c for c in raw_test.columns if c != 'label']
reference = pd.read_csv(f'data/{SCALING_REFERENCE}')[gene_columns].to_numpy()
low, high = reference.min(axis=0), reference.max(axis=0)
X_test = (raw_test[gene_columns].to_numpy() - low) / (high - low)
y_test = raw_test['label'].to_numpy(dtype=np.intp)
assert np.abs(X_test - pd.read_csv(f'data/{SCALED_TEST_FILE}')[gene_columns].to_numpy()).max() < 1e-9

ids = pd.read_csv(f'data/{TEST_IDS_FILE}')
assert np.allclose(ids[gene_columns].to_numpy(float), raw_test[gene_columns].to_numpy(float)), 'ID file: other patients'
assert (ids['label'].to_numpy() == y_test).all() and ids['SampleID'].is_unique
sample_ids = ids['SampleID'].to_numpy()
X_train, y_train = DATA[TRAIN_FILE]
print(f'Test: {len(y_test)} patients ({(y_test == 1).sum()} positive, {(y_test == 0).sum()} negative). '
      f'Train: {len(y_train)} rows.')


def fit_predict(model):
    # Trains the model on the training partition restricted to its genes; predicted labels and probability of class 1
    genes = ast.literal_eval(model['Subset'])
    classifier = make_classifier(model['Classifier'])
    classifier.fit(X_train[:, genes], y_train)
    return classifier.predict(X_test[:, genes]), classifier.predict_proba(X_test[:, genes])[:, 1]


def confusion(y_pred):
    tp = int(((y_test == 1) & (y_pred == 1)).sum())
    fn = int(((y_test == 1) & (y_pred == 0)).sum())
    tn = int(((y_test == 0) & (y_pred == 0)).sum())
    fp = int(((y_test == 0) & (y_pred == 1)).sum())
    return tp, fn, tn, fp


models = pairs.drop_duplicates('Model').set_index('Model')
predictions = {model: fit_predict(row) for model, row in models[models['Signature'] == 'MH'].iterrows()}

known = pd.read_csv(KNOWN_RESULTS_FILE).set_index('name')
check = []
for model, row in models[models['Signature'] == 'MH'].iterrows():
    tp, fn, tn, fp = confusion(predictions[model][0])
    published = known.loc[f"{row['Meta']}_{row['OF']}"]
    check.append({'Finalist': row['Finalist'], 'Classifier': row['Classifier'], 'TP': tp, 'FN': fn, 'TN': tn, 'FP': fp,
                  'Published (TP, FN, TN, FP)': (int(published['tp']), int(published['fn']), int(published['tn']), int(published['fp'])),
                  'Reproduced': (tp, fn, tn, fp) == (published['tp'], published['fn'], published['tn'], published['fp'])})
check = pd.DataFrame(check)
display(check)
assert check['Reproduced'].all(), 'the procedure does not reproduce the published results of the finalists'
print(f'Check passed: the {len(check)} finalists reproduce their published confusion counts.')

Test: 30 patients (10 positive, 20 negative). Train: 356 rows.


,Finalist,Classifier,TP,FN,TN,FP,"Published (TP, FN, TN, FP)",Reproduced
0,DE + aMI,Naive,9,1,8,12,"(9, 1, 8, 12)",True
1,DE + aSU,RF estimators: 60,10,0,17,3,"(10, 0, 17, 3)",True
2,DE + aGR,RF estimators: 60,10,0,13,7,"(10, 0, 13, 7)",True
3,DE + aReliefF,Knn k = 3,9,1,14,6,"(9, 1, 14, 6)",True
4,DE + aCHI2,RF estimators: 100,9,1,13,7,"(9, 1, 13, 7)",True
5,GA + aMI,RF estimators: 60,5,5,14,6,"(5, 5, 14, 6)",True
6,GA + aSU,RF estimators: 80,8,2,14,6,"(8, 2, 14, 6)",True
7,GA + aGR,RF estimators: 80,6,4,13,7,"(6, 4, 13, 7)",True
8,GA + aReliefF,Knn k = 3,10,0,14,6,"(10, 0, 14, 6)",True
9,GA + aCHI2,SVM (Poly),7,3,11,9,"(7, 3, 11, 9)",True


Check passed: the 15 finalists reproduce their published confusion counts.


In [9]:
# Sensitivity to the gene order: the random forests whose genes are not in ascending order, refitted with sorted genes
order_rows = []
for model, row in models[models['Signature'] == 'MH'].iterrows():
    genes = ast.literal_eval(row['Subset'])
    if not row['Classifier'].lower().startswith('rf') or genes == sorted(genes):
        continue
    y_sorted, _ = fit_predict({**row, 'Subset': str(sorted(genes))})
    as_file, as_sorted = confusion(predictions[model][0]), confusion(y_sorted)
    order_rows.append({'Finalist': row['Finalist'], 'Classifier': row['Classifier'],
                       'File order (TP, FN, TN, FP)': as_file, 'Ascending order (TP, FN, TN, FP)': as_sorted,
                       'Patients classified differently': int((predictions[model][0] != y_sorted).sum())})
display(pd.DataFrame(order_rows))

,Finalist,Classifier,"File order (TP, FN, TN, FP)","Ascending order (TP, FN, TN, FP)",Patients classified differently
0,SA + aMI,RF estimators: 100,"(7, 3, 10, 10)","(8, 2, 11, 9)",2
1,SA + aSU,RF estimators: 60,"(6, 4, 14, 6)","(5, 5, 14, 6)",5
2,SA + aGR,RF estimators: 80,"(5, 5, 11, 9)","(4, 6, 11, 9)",5
3,SA + aCHI2,RF estimators: 100,"(7, 3, 13, 7)","(8, 2, 13, 7)",3


## 6. Freeze

The configurations in `top10_pairs.csv` must be fixed before any top-10 model is tested. This cell stops the notebook
unless the file is committed in git and unchanged since that commit; running stage 2 again rewrites it identically.

**To continue:** commit `top10_pairs.csv` (with the other stage 1 and 2 outputs), then run all cells again. The commit
is recorded in `test_run.json`.

In [10]:
def git(*args):
    return subprocess.run(['git', *args], capture_output=True, text=True)


tracked = git('ls-files', '--error-unmatch', PAIRS_FILE).returncode == 0
unchanged = tracked and git('diff', '--quiet', 'HEAD', '--', PAIRS_FILE).returncode == 0
if not unchanged:
    raise RuntimeError(f'FREEZE: commit {PAIRS_FILE} before the top-10 models are tested, then run all cells again '
                       f'(committed: {tracked}; unchanged since the commit: {unchanged}).')
freeze_commit, freeze_date = git('log', '-1', '--format=%H %cI', '--', PAIRS_FILE).stdout.split()
with open(PAIRS_FILE, 'rb') as file:
    pairs_sha256 = hashlib.sha256(file.read()).hexdigest()
print(f'Configurations frozen in commit {freeze_commit} ({freeze_date}).')

Configurations frozen in commit c056a16a1e067590c5cd2ce9e697a138b6a0efb0 (2026-11-27T13:21:58+01:00).


## 7. Stage 3: test evaluation

The top-10 models are trained and tested (each distinct model once). Then, for all the models:

* **Counts and metrics.** TP, FN, TN and FP on the 30 patients, and F1, Sn, Sp, ACC and PRE.
* **95% confidence intervals.** Wilson score interval for Sn, Sp, ACC and PRE. Percentile bootstrap for F1, drawing
  patients with replacement separately among the 10 positives and the 20 negatives, so every resample keeps the
  class counts (100,000 resamples, seed 45). This is the method of `confidence_intervals.py`, with the same resamples.
* **Paired differences.** For each finalist, MH minus Top-10 in F1, Sn, Sp and ACC, with a bootstrap interval in
  which both models are scored on the same resampled patients, and the exact McNemar test on the patients where only
  one of the two is right, with Benjamini–Hochberg correction over the 15 pairs.

These are observed differences with their uncertainty, in an exploratory comparison; they do not show that one
approach is better in general.

In [11]:
for model, row in models[models['Signature'] == 'TOP10'].iterrows():
    predictions[model] = fit_predict(row)

# A top 10 equal to the signature is the same model: it must give the same predictions
for _, row in pairs[pairs['Same_as_MH'] & (pairs['Signature'] == 'TOP10')].iterrows():
    mh_model = pairs.loc[(pairs['Pair'] == row['Pair']) & (pairs['Signature'] == 'MH'), 'Model'].iloc[0]
    assert (predictions[row['Model']][0] == predictions[mh_model][0]).all(), f"{row['Finalist']}: different predictions"
print(f'{len(predictions)} models tested on {len(y_test)} patients.')

26 models tested on 30 patients.


In [12]:
# Resampled patients, shared by all models: one array of positives and one of negatives, one row per resample
rng = np.random.default_rng(BOOTSTRAP_SEED)
positives, negatives = np.flatnonzero(y_test == 1), np.flatnonzero(y_test == 0)
draw_pos = positives[rng.integers(0, len(positives), (RESAMPLES, len(positives)))]
draw_neg = negatives[rng.integers(0, len(negatives), (RESAMPLES, len(negatives)))]


def metric_values(tp, fn, tn, fp):
    # Works with numbers and with arrays of resampled counts. PRE is NaN when nothing is predicted positive
    with np.errstate(divide='ignore', invalid='ignore'):
        return {'F1': 2 * tp / (2 * tp + fp + fn), 'Sn': tp / (tp + fn), 'Sp': tn / (tn + fp),
                'ACC': (tp + tn) / (tp + fn + tn + fp), 'PRE': np.divide(tp, tp + fp)}


def resampled(y_pred):
    tp = (y_pred[draw_pos] == 1).sum(axis=1)
    tn = (y_pred[draw_neg] == 0).sum(axis=1)
    return metric_values(tp, draw_pos.shape[1] - tp, tn, draw_neg.shape[1] - tn)


def wilson(successes, total):
    if total == 0:
        return np.nan, np.nan
    interval = binomtest(int(successes), int(total)).proportion_ci(confidence_level=CONFIDENCE, method='wilson')
    return interval.low, interval.high


def percentile_interval(values):
    tail = 50 * (1 - CONFIDENCE)
    low, high = np.percentile(values, [tail, 100 - tail])
    return low, high


by_model = {}
for model, (y_pred, _) in predictions.items():
    tp, fn, tn, fp = confusion(y_pred)
    values = metric_values(tp, fn, tn, fp)
    limits = {'F1': percentile_interval(resampled(y_pred)['F1']), 'Sn': wilson(tp, tp + fn), 'Sp': wilson(tn, tn + fp),
              'ACC': wilson(tp + tn, len(y_test)), 'PRE': wilson(tp, tp + fp)}
    record = {'TP': tp, 'FN': fn, 'TN': tn, 'FP': fp}
    for metric in METRICS:
        record[metric] = values[metric]
        record[f'{metric}_low'], record[f'{metric}_high'] = limits[metric]
    by_model[model] = record

columns = ['Pair', 'Finalist', 'Meta', 'OF', 'Classifier', 'Signature', 'Model', 'Length', 'Same_as_MH', 'Shared_by']
test_metrics = pd.concat([pairs[columns].reset_index(drop=True),
                          pd.DataFrame([by_model[model] for model in pairs['Model']])], axis=1)

differences = []
for _, group in pairs.groupby('Pair', sort=True):
    mh, top = group[group['Signature'] == 'MH'].iloc[0], group[group['Signature'] == 'TOP10'].iloc[0]
    y_mh, y_top = predictions[mh['Model']][0], predictions[top['Model']][0]
    only_mh = int(((y_mh == y_test) & (y_top != y_test)).sum())
    only_top = int(((y_mh != y_test) & (y_top == y_test)).sum())
    p_value = 1.0 if only_mh + only_top == 0 else binomtest(only_mh, only_mh + only_top, 0.5).pvalue
    resampled_mh, resampled_top = resampled(y_mh), resampled(y_top)
    for metric in DIFFERENCE_METRICS:
        low, high = percentile_interval(resampled_mh[metric] - resampled_top[metric])
        differences.append({'Pair': mh['Pair'], 'Finalist': mh['Finalist'], 'Classifier': mh['Classifier'],
                            'Metric': metric, 'MH': by_model[mh['Model']][metric], 'TOP10': by_model[top['Model']][metric],
                            'Delta': by_model[mh['Model']][metric] - by_model[top['Model']][metric], 'Low': low, 'High': high,
                            'Only_MH_correct': only_mh, 'Only_TOP10_correct': only_top, 'McNemar_p': p_value})
differences = pd.DataFrame(differences)
p_by_pair = differences.drop_duplicates('Pair').set_index('Pair')['McNemar_p']
adjusted = dict(zip(p_by_pair.index, false_discovery_control(p_by_pair.to_numpy(), method='bh')))
differences['McNemar_p_BH'] = differences['Pair'].map(adjusted)

test_predictions = pd.concat([pd.DataFrame({'Model': model, 'SampleID': sample_ids, 'y_true': y_test, 'y_pred': y_pred,
                                            'p_positive': probability})
                              for model, (y_pred, probability) in predictions.items()], ignore_index=True)
run_info = {
    'pairs_file': PAIRS_FILE, 'pairs_sha256': pairs_sha256, 'freeze_commit': freeze_commit, 'freeze_date': freeze_date,
    'train_file': TRAIN_FILE, 'test_file': TEST_FILE, 'patients': int(len(y_test)),
    'positives': int((y_test == 1).sum()), 'negatives': int((y_test == 0).sum()), 'models': len(predictions),
    'confidence': CONFIDENCE, 'resamples': RESAMPLES, 'bootstrap_seed': BOOTSTRAP_SEED,
    'scikit_learn': sklearn.__version__, 'numpy': np.__version__,
}
test_predictions.to_csv(PREDICTIONS_FILE, index=False)
test_metrics.to_csv(METRICS_FILE, index=False)
differences.to_csv(DIFFERENCES_FILE, index=False)
with open(RUN_FILE, 'w') as file:
    json.dump(run_info, file, indent=2)
print('Written to', ', '.join([PREDICTIONS_FILE, METRICS_FILE, DIFFERENCES_FILE, RUN_FILE]))


def text(value, low, high):
    return f'{value:.2f} [{low:.2f}, {high:.2f}]'


summary = test_metrics[['Finalist', 'Classifier', 'Signature', 'Length']].copy()
for metric in DIFFERENCE_METRICS:
    summary[metric] = [text(r[metric], r[f'{metric}_low'], r[f'{metric}_high']) for _, r in test_metrics.iterrows()]
display(summary)
display(differences[differences['Metric'] == 'F1'][['Finalist', 'MH', 'TOP10', 'Delta', 'Low', 'High',
                                                     'Only_MH_correct', 'Only_TOP10_correct', 'McNemar_p', 'McNemar_p_BH']])

Written to experiment_results/top10_ranking_vs_mh/test_predictions.csv, experiment_results/top10_ranking_vs_mh/test_metrics.csv, experiment_results/top10_ranking_vs_mh/test_differences.csv, experiment_results/top10_ranking_vs_mh/test_run.json


,Finalist,Classifier,Signature,Length,F1,Sn,Sp,ACC
0,DE + aMI,Naive,MH,10,"0.58 [0.45, 0.69]","0.90 [0.60, 0.98]","0.40 [0.22, 0.61]","0.57 [0.39, 0.73]"
1,DE + aMI,Naive,TOP10,10,"0.48 [0.30, 0.64]","0.70 [0.40, 0.89]","0.40 [0.22, 0.61]","0.50 [0.33, 0.67]"
2,DE + aSU,RF estimators: 60,MH,10,"0.87 [0.77, 1.00]","1.00 [0.72, 1.00]","0.85 [0.64, 0.95]","0.90 [0.74, 0.97]"
3,DE + aSU,RF estimators: 60,TOP10,10,"0.57 [0.32, 0.78]","0.60 [0.31, 0.83]","0.75 [0.53, 0.89]","0.70 [0.52, 0.83]"
4,DE + aGR,RF estimators: 60,MH,10,"0.74 [0.65, 0.87]","1.00 [0.72, 1.00]","0.65 [0.43, 0.82]","0.77 [0.59, 0.88]"
5,DE + aGR,RF estimators: 60,TOP10,10,"0.74 [0.65, 0.87]","1.00 [0.72, 1.00]","0.65 [0.43, 0.82]","0.77 [0.59, 0.88]"
6,DE + aReliefF,Knn k = 3,MH,18,"0.72 [0.57, 0.87]","0.90 [0.60, 0.98]","0.70 [0.48, 0.85]","0.77 [0.59, 0.88]"
7,DE + aReliefF,Knn k = 3,TOP10,10,"0.67 [0.52, 0.80]","0.90 [0.60, 0.98]","0.60 [0.39, 0.78]","0.70 [0.52, 0.83]"
8,DE + aCHI2,RF estimators: 100,MH,10,"0.69 [0.54, 0.83]","0.90 [0.60, 0.98]","0.65 [0.43, 0.82]","0.73 [0.56, 0.86]"
9,DE + aCHI2,RF estimators: 100,TOP10,10,"0.56 [0.35, 0.75]","0.70 [0.40, 0.89]","0.60 [0.39, 0.78]","0.63 [0.46, 0.78]"


,Finalist,MH,TOP10,Delta,Low,High,Only_MH_correct,Only_TOP10_correct,McNemar_p,McNemar_p_BH
0,DE + aMI,0.580645,0.482759,0.097887,0.000000,0.259048,2,0,0.500000,0.990767
4,DE + aSU,0.869565,0.571429,0.298137,0.074074,0.571429,7,1,0.070312,0.527344
8,DE + aGR,0.740741,0.740741,0.000000,0.000000,0.000000,0,0,1.000000,1.000000
12,DE + aReliefF,0.720000,0.666667,0.053333,-0.140741,0.248276,5,3,0.726562,0.990767
16,DE + aCHI2,0.692308,0.560000,0.132308,-0.028986,0.333333,5,2,0.453125,0.990767
20,GA + aMI,0.476190,0.640000,-0.163810,-0.465241,0.104348,3,5,0.726562,0.990767
24,GA + aSU,0.666667,0.545455,0.121212,-0.128696,0.397059,5,3,0.726562,0.990767
28,GA + aGR,0.521739,0.769231,-0.247492,-0.500000,-0.050707,1,6,0.125000,0.625000
32,GA + aReliefF,0.769231,0.666667,0.102564,-0.064103,0.287879,6,3,0.507812,0.990767
36,GA + aCHI2,0.538462,0.500000,0.038462,-0.144928,0.222222,5,3,0.726562,0.990767


### Stage 3 output columns

`test_predictions.csv`: one row per model and patient.

| Column | Meaning |
|---|---|
| `Model` | Model name, as in `top10_pairs.csv` |
| `SampleID` | Patient identifier from `test_rtf_with_ID_original.csv` |
| `y_true`, `y_pred` | True and predicted label (1 = positive class) |
| `p_positive` | Probability of the positive class given by the classifier (not seeded for SVM) |

`test_metrics.csv`: one row per pair and signature (30 rows); the input of the LaTeX table.

| Column | Meaning |
|---|---|
| `Pair` … `Shared_by` | As in `top10_pairs.csv` |
| `TP`, `FN`, `TN`, `FP` | Confusion counts on the 30 patients |
| `F1`, `Sn`, `Sp`, `ACC`, `PRE` | Metric values |
| `<metric>_low`, `<metric>_high` | 95% limits: bootstrap for F1, Wilson for the others |

`test_differences.csv`: one row per pair and metric (15 × 4 rows).

| Column | Meaning |
|---|---|
| `Pair`, `Finalist`, `Classifier` | As above |
| `Metric` | F1, Sn, Sp or ACC |
| `MH`, `TOP10`, `Delta` | Value with the signature, with the top 10, and MH − Top-10 (positive: the signature did better) |
| `Low`, `High` | 95% limits of `Delta`, paired bootstrap |
| `Only_MH_correct`, `Only_TOP10_correct` | Patients classified correctly by only one of the two models |
| `McNemar_p`, `McNemar_p_BH` | Exact McNemar p-value, and corrected by Benjamini–Hochberg over the 15 pairs |

`test_run.json`: commit and date of the freeze, SHA-256 of `top10_pairs.csv`, data files, numbers of patients and
models, interval settings, and the scikit-learn and NumPy versions.

**Next:** `python latex_tables/top10_ranking_vs_mh_table.py` writes the table from `test_metrics.csv`.

## 8. Summary of the experiment

**Question.** Eq. (11) averages per-gene scores, so with a minimum of 10 genes the 10 best-ranked genes should
maximize it. Does the metaheuristic search add anything over that direct top-10 ranking, on the objective and on the
test set?

**What was done.**

| Step | Section | What | Result |
|---|---|---|---|
| Top 10 | 2–3 | For each of the five objectives, the 10 best genes on the training partition (356 rows): Weka `Ranker -N 10` with the objective's evaluator; for aMI, its scikit-learn scores sorted | The scores equal those of `ofunctions.allObjectiveFunctionWeka`. No tie at the cut, so the tie rule was not needed |
| Signatures vs top 10 | 4 | Genes in common and Eq. (11) for each of the 15 finalists | See *Objective* below |
| Configurations | 4, 6 | 30 models (15 signatures and 15 top 10, 26 distinct): each top 10 keeps its finalist's classifier, parameters and seed | Frozen in git (commit `c056a16`) before any top-10 model was tested |
| Check | 5 | The 15 finalists re-run with this procedure | Their published confusion counts are reproduced exactly |
| Test | 7 | The 11 new top-10 models on the same 30 patients; 95% intervals, paired differences and McNemar with Benjamini–Hochberg (BH) | See *Test* below. LaTeX table: label `tableTop10RankingVsMH` |

**Objective (stage 2).**

* aMI, aSU, aGR and aCHI2: on the training partition the top 10 has at least the merit of all 12 signatures, as
  expected for fixed per-gene scores. The implementation adds no constraint that changes this.
* Only DE–aGR is the same subset as its top 10. The other signatures share 2–9 genes with it.
* On the 396-row file the metaheuristics searched, the four DE signatures of these objectives are exactly that file's
  top 10. GA and SA did not reach it.
* aReliefF, where a gene's weight depends on the other genes: DE's 18-gene signature has a
  higher Eq. (11) than the top 10 (0.0140 vs 0.0130).
  GA's and SA's do not (0.0129 and 0.0126).

**Test (stage 3).** F1-score on the 30 patients. Δ = MH − Top-10 (positive: the signature did better); its interval
is the paired bootstrap. Full metrics in `test_metrics.csv` and `test_differences.csv`.

| Pair | Finalist | Classifier | L (MH) | Genes in common | F1 MH | F1 Top-10 | Δ F1 [95% CI] | McNemar p (BH) |
|---|---|---|---|---|---|---|---|---|
| 1 | DE–aMI | Naive | 10 | 7 of 10 | 0.58 | 0.48 | +0.10 [0.00, 0.26] | 0.99 |
| 2 | DE–aSU | RF estimators: 60 | 10 | 7 of 10 | 0.87 | 0.57 | +0.30 [0.07, 0.57] | 0.53 |
| 3 | DE–aGR | RF estimators: 60 | 10 | 10 of 10 | 0.74 | 0.74 | +0.00 [0.00, 0.00] | 1.00 |
| 4 | DE–aReliefF | Knn k = 3 | 18 | 9 of 10 | 0.72 | 0.67 | +0.05 [−0.14, 0.25] | 0.99 |
| 5 | DE–aCHI2 | RF estimators: 100 | 10 | 7 of 10 | 0.69 | 0.56 | +0.13 [−0.03, 0.33] | 0.99 |
| 6 | GA–aMI | RF estimators: 60 | 10 | 5 of 10 | 0.48 | 0.64 | −0.16 [−0.47, 0.10] | 0.99 |
| 7 | GA–aSU | RF estimators: 80 | 10 | 6 of 10 | 0.67 | 0.55 | +0.12 [−0.13, 0.40] | 0.99 |
| 8 | GA–aGR | RF estimators: 80 | 10 | 6 of 10 | 0.52 | 0.77 | −0.25 [−0.50, −0.05] | 0.62 |
| 9 | GA–aReliefF | Knn k = 3 | 11 | 5 of 10 | 0.77 | 0.67 | +0.10 [−0.06, 0.29] | 0.99 |
| 10 | GA–aCHI2 | SVM (Poly) | 10 | 6 of 10 | 0.54 | 0.50 | +0.04 [−0.14, 0.22] | 0.99 |
| 11 | SA–aMI | RF estimators: 100 | 10 | 2 of 10 | 0.52 | 0.62 | −0.10 [−0.32, 0.11] | 0.99 |
| 12 | SA–aSU | RF estimators: 60 | 10 | 6 of 10 | 0.55 | 0.57 | −0.03 [−0.31, 0.26] | 1.00 |
| 13 | SA–aGR | RF estimators: 80 | 10 | 5 of 10 | 0.42 | 0.77 | −0.35 [−0.60, −0.15] | 0.12 |
| 14 | SA–aReliefF | Knn k = 5 | 10 | 6 of 10 | 0.69 | 0.64 | +0.05 [−0.07, 0.19] | 1.00 |
| 15 | SA–aCHI2 | RF estimators: 100 | 10 | 5 of 10 | 0.58 | 0.56 | +0.02 [−0.22, 0.27] | 1.00 |

* F1 is higher with the signature in 9 pairs, with the top 10 in 5, and equal in 1 (DE–aGR, the same model).
* The paired 95% interval excludes 0 in three pairs: DE–aSU (favours the signature), GA–aGR and SA–aGR (favour the top 10).
* No McNemar test is significant after BH correction (smallest corrected p = 0.12, SA–aGR).

**How to read it.**

1. **The two signatures were selected on different data.** The metaheuristics searched `normal_rtf.csv` (396 rows),
   which contains all 30 test patients; the top 10 uses only the training partition. The DE signatures of the
   separable objectives are the top 10 of those 396 rows, so their difference from the training top 10 comes only from
   the extra rows, not from the search. The DE advantages on test (F1 +0.30 in aSU, +0.13 in aCHI2, +0.10 in aMI)
   cannot be attributed to the search, and they may be inflated because those genes were chosen on data that
   included the test patients. The GA and SA signatures were selected on the same data, so the same bias favours them.
2. **Exploratory.** The finalists' test results were already known when this comparison was designed. The
   configurations were frozen before the new results were computed, and nothing was adjusted afterwards.
3. **Size of arbitrary differences.** Changing only the column order of four random forests changes 2–5 of the 30
   predictions (section 5). Several differences in the table are of that size.
4. **Small test set.** 30 patients (10 positive) give wide intervals.

**What the search adds.** On the objective, nothing for aMI, aSU, aGR and aCHI2: the ranking gives the optimum,
DE found that same optimum, and GA and SA fell short of it. For aReliefF, DE found a subset with a higher Eq. (11) than
the ranking, but its test F1 is not distinguishable from the top 10's. On the test set the differences go both ways
and most are within their uncertainty; given point 1, they cannot show an advantage of the search.

**Not covered here.** Ranking inside each cross-validation fold, re-running the metaheuristics on the training
partition only, and the computational cost of search versus ranking.

**Outputs** (in `experiment_results/top10_ranking_vs_mh/`): `rankings.csv` and `top10_subsets.csv` (stage 1);
`signatures_vs_top10.csv` and `top10_pairs.csv` (stage 2, frozen); `test_predictions.csv`, `test_metrics.csv`,
`test_differences.csv` and `test_run.json` (stage 3). LaTeX table: `latex_tables/output/top10_ranking_vs_mh.tex`.